# 04 · Retrieve — 09 Multi-query retrieval

**Ported from `terrier-ta/course_rag.py`'s `retrieve_multi_query`. Nothing in this cookbook runs more than one query formulation per question -- this is also the cheapest partial answer to the zero-retrieval problem: a badly-phrased query returning nothing is a different failure from a corpus that genuinely lacks the answer, and running several phrasings tells them apart.**

Scope, per the copy plan: the retrieval and merge only. Generating the
alternative phrasings with a model (query rewriting) is a separate,
out-of-scope capability -- the phrasings below are hand-written for the
demonstration, not produced by an LLM.

## What this notebook demonstrates

| Name | What it does | Example |
|---|---|---|
| `retrieve` | The single-query search this notebook builds on (same shape as notebook 08, kept self-contained here) | `retrieve("bio201", "mitochondria")` |
| `retrieve_multi_query` | Runs several query phrasings, merges results by max score per chunk id | `retrieve_multi_query("bio201", ["mitochondria", "powerhouse of the cell"])` |


In [ ]:
import sys
from pathlib import Path

_p = Path.cwd().resolve()
for _ in range(6):
    if (_p / "nbio.py").is_file():
        sys.path.insert(0, str(_p))
        break
    _p = _p.parent
else:
    raise RuntimeError("could not locate nbio.py above the current directory")

import nbio

repo_root = nbio.bootstrap()

## Step 1 — a course store where two phrasings of the same question score very differently

This is built to make the point concrete: the stored chunk uses the word
"mitochondria" and the technical term "oxidative phosphorylation", never
the informal phrase "powerhouse of the cell" that a real question might
use instead. A hash-based embedding (same offline stand-in as every other
notebook in this stage) has no notion of synonymy, so a query using only
the informal phrase scores this chunk much lower than a query using its
own vocabulary -- the exact gap multi-query retrieval exists to cover.

In [ ]:
import hashlib
import math


def hash_embed(text: str, dim: int = 384) -> list[float]:
    vec = [0.0] * dim
    tokens = (text or "").lower().split()
    if not tokens:
        return vec
    for tok in tokens:
        h = int(hashlib.sha256(tok.encode("utf-8")).hexdigest(), 16)
        idx = h % dim
        sign = 1.0 if (h >> 8) & 1 else -1.0
        vec[idx] += sign
    norm = math.sqrt(sum(v * v for v in vec)) or 1.0
    return [v / norm for v in vec]


def cosine(a: list[float], b: list[float]) -> float:
    if not a or not b or len(a) != len(b):
        return 0.0
    dot = sum(x * y for x, y in zip(a, b))
    na = math.sqrt(sum(x * x for x in a)) or 1.0
    nb = math.sqrt(sum(x * x for x in b)) or 1.0
    return dot / (na * nb)


COURSE_STORE = [
    {"chunk_id": "bio201::c0", "text": "Mitochondria perform oxidative phosphorylation, converting nutrients into ATP."},
    {"chunk_id": "bio201::c1", "text": "Photosynthesis converts light energy into chemical energy stored in glucose."},
]
for ch in COURSE_STORE:
    ch["embedding"] = hash_embed(ch["text"])

print(f"{len(COURSE_STORE)} chunks seeded")

## Step 2 — `retrieve`, one query at a time

Same shape as notebook 08's `search_local` -- kept as its own small
function here rather than imported, since notebooks in this repo are
self-contained and don't share runtime state with each other.

In [ ]:
def retrieve(query: str, top_k: int = 5) -> list[dict]:
    qvec = hash_embed(query)
    scored = [{**ch, "score": cosine(qvec, ch["embedding"])} for ch in COURSE_STORE]
    scored.sort(key=lambda x: -x["score"])
    return scored[:top_k]


formal_query = "What is the role of mitochondria in oxidative phosphorylation?"
informal_query = "what is the powerhouse of the cell"

print("formal phrasing, top result:")
for r in retrieve(formal_query, top_k=1):
    print(f"  score={r['score']:.3f}  {r['text'][:60]!r}")

print("\ninformal phrasing, top result:")
for r in retrieve(informal_query, top_k=1):
    print(f"  score={r['score']:.3f}  {r['text'][:60]!r}")

## Step 3 — `retrieve_multi_query`: run both, merge by max score per chunk

Ported from `retrieve_multi_query`: run every phrasing through `retrieve`,
and for each chunk id seen more than once, keep the *higher* of the scores
it received -- so a chunk that one phrasing missed and another phrasing
found still surfaces, at the score of the phrasing that actually found
it.

In [ ]:
def retrieve_multi_query(queries: list[str], top_k_per_query: int = 5) -> list[dict]:
    merged: dict[str, dict] = {}
    for q in queries:
        q = (q or "").strip()
        if not q:
            continue
        for ch in retrieve(q, top_k=top_k_per_query):
            key = ch["chunk_id"]
            prev = merged.get(key)
            if prev is None or ch["score"] > prev["score"]:
                merged[key] = ch
    return sorted(merged.values(), key=lambda x: -x["score"])


single_query_result = retrieve(informal_query, top_k=1)
multi_query_result = retrieve_multi_query([informal_query, formal_query], top_k_per_query=2)

print("informal phrasing ALONE, top result:")
for r in single_query_result:
    print(f"  score={r['score']:.3f}  {r['text'][:60]!r}")

print("\ninformal + formal phrasing MERGED, top result:")
for r in multi_query_result[:1]:
    print(f"  score={r['score']:.3f}  {r['text'][:60]!r}")

assert multi_query_result[0]["score"] >= single_query_result[0]["score"], (
    "merging in the formal phrasing should never score the same top chunk lower"
)
print()
print("confirmed: the merged, multi-query score is at least as good as the single informal query's own best score")

## Why this matters for the zero-retrieval problem

`06-bench`'s leaderboard names an open task: on six identical runs of the
same question, five retrieved real papers and one retrieved none. Before
concluding a corpus lacks an answer, multi-query retrieval is the cheap
first check -- if a *second* phrasing of the same question finds
something the first one didn't, the failure was the query, not the
corpus. It doesn't diagnose *why* one phrasing failed; it only tells that
failure apart from a genuinely empty corpus, which is the distinction the
open task actually needs.